In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
import joblib 
import numpy as np

# 1. Tải tập dữ liệu từ quá trình Trace-driven Simulation
df = pd.read_csv('../trace_driven_simulation/data/simulation_data_198.csv')
TAU_SLO = 200
df_clean = df[df['response_time'] <= TAU_SLO].copy()
df_clean['slo_threshold'] = TAU_SLO

# Features: requests, SLO_threshold | Label: vms_count
X = df_clean[['requests', 'slo_threshold']]
y = df_clean['vms_count']

In [10]:
# 2. Chia dữ liệu (Random shuffle để bao phủ dải tải)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
X_train

,requests,slo_threshold
71,147000,200
175,355000,200
156,317000,200
163,331000,200
161,327000,200
...,...,...
90,185000,200
135,275000,200
18,41000,200
117,239000,200


In [ ]:
# 3. Định nghĩa các mô hình
model_M = DecisionTreeRegressor()


In [13]:
# 4. Huấn luyện và đánh giá
model_M.fit(X_train, y_train)
y_pred = model_M.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
print(f"Root Mean Squared Error (RMSE) : {rmse:.4f}")


Root Mean Squared Error (RMSE) : 0.1768


In [14]:
# --- TẠO BẢNG SO SÁNH THỰC TẾ VÀ DỰ ĐOÁN ---
comparison_df = pd.DataFrame({
    'Thứ tự (Phút)': df_clean.loc[X_test.index, 'minute'].values,  # Lấy chính xác phút từ file gốc
    'Số requests': X_test['requests'].values,
    'Thực tế (vm_count)': y_test,
    # Làm tròn kết quả dự đoán thành số nguyên vì số lượng VM không thể là số lẻ
    'Dự đoán (predicted_vm)': y_pred.round().astype(int) 
})

# In ra 15 dòng đầu tiên để kiểm tra trực quan
print("BẢNG SO SÁNH SỐ LƯỢNG MÁY CHỦ (THỰC TẾ vs DỰ ĐOÁN):")
print(comparison_df.head(15))

BẢNG SO SÁNH SỐ LƯỢNG MÁY CHỦ (THỰC TẾ vs DỰ ĐOÁN):
     Thứ tự (Phút)  Số requests  Thực tế (vm_count)  Dự đoán (predicted_vm)
122            123       249000                  27                      27
88              89       181000                  20                      20
104            105       213000                  23                      23
97              98       199000                  22                      22
145            146       295000                  32                      32
37              38        79000                   9                       9
119            120       243000                  26                      26
168            169       341000                  37                      37
118            119       241000                  26                      26
177            178       359000                  39                      39
24              25        53000                   6                       6
114            115       233000     

In [15]:
# 7. Lưu trữ mô hình để dùng cho Algorithm 2
model_filename = './model/predictive_autoscaling_model_M_1.pkl'
joblib.dump(model_M, model_filename)
print(f"Đã lưu mô hình thành công tại: {model_filename}")

Đã lưu mô hình thành công tại: ./model/predictive_autoscaling_model_M_1.pkl


In [16]:
y_pred

array([27., 20., 23., 22., 32.,  9., 26., 37., 26., 39.,  6., 25.,  5.,
       35.,  7.,  9., 33., 28., 42., 22., 28.,  6.,  4.,  3., 10., 29.,
       19., 16., 15., 27., 13., 41.])